In [1]:
%cd D:/StyleTransferAI/StyleTransferAI_AdaIN/StyleTransferAI-based-on-AdaIN/style_transfer_api/
# %load_ext gradio

import gradio as gr
import numpy as np
from PIL import Image
import asyncio
from utils.requests import Request, Response
from utils.file import File
import uuid
import websockets
from websockets.asyncio.client import connect
import json
from pathlib import Path
from nicegui import ui

D:\StyleTransferAI\StyleTransferAI_AdaIN\StyleTransferAI-based-on-AdaIN\style_transfer_api


### TODO
- ~~Transmit all interface components as generation function input to access their real value~~
- Transfer the interface to NiceGUI
- Handle request cancellation
- Use Requests and Responses objects to communicate
- Auto adjust style weights to have them summed to 1
- Reduce Add style button width below 3 style images
- Add titles to the interface components
- Provide a French / English translation

In [2]:
API_URL = "ws://localhost:8000/generate"

class StyleBlock :
    def __init__(self, visible = False):
        with ui.column() as self.block :
            self.img = ui.image()
            self.weight = ui.slider(label="Weight", min=0, max=1, value=1)
            self.scale = ui.slider(label="Spatial scaling", min=0.1, max=3, value=1)
            self.rmv_btn = ui.button(text="Remove style", visible=False)
            # self.true_res_img = gr.Image(render=False)
        # self.img.input(self.resize, inputs=self.img, outputs=self.img) # Image is always resized to a fixed height to make sure all style blocks are correctly aligned

    # def resize(self, img) :
    #     self.true_res_img.value = img # Keep the original image as it will be used for computation
    #     if img is None :
    #         return gr.update(value=None)
    #     # img = Image.fromarray(img)
    #     img = Image.fr
    #     new_width = int((IMG_HEIGHT / img.height) * img.width)
    #     new_img = img.resize((new_width, IMG_HEIGHT))
    #     return gr.update(value=new_img)

    def list(self) :
        return [self.img, self.weight, self.scale] #, self.true_res_img]
    
    def reset(self) :
        return (gr.update(value=None), gr.update(value=1), gr.update(value=1)) #, gr.update(value=None))
    
    def copy(self, new_img, new_weight, new_scale) : #, new_true_res_img) :
        return (gr.update(value = new_img), gr.update(value = new_weight), gr.update(value = new_scale)) #, gr.update(value = new_true_res_img))


class ParamsBlock :
    def __init__(self):
        with gr.Blocks() as self.block :
            self.alpha = gr.Slider(0, 1, value = 1, label="Importance of stylization", interactive=True)
            with gr.Row(equal_height=True) :
                with gr.Column(scale=2) :
                    self.pres_col = gr.Checkbox(label="Preserve colors", value=False)
                    # sep = gr.Markdown("---")
                    with gr.Group() :
                        self.res_size = gr.Slider(label="Resize size", minimum=256, maximum=2000, value=1000, interactive=True)
                        self.keep_ratio = gr.Checkbox(label="Keep aspect ratio", value=False)
                with gr.Column(scale=1) :
                    self.patches = gr.Checkbox(label="Work with patches", value=False)
                    self.patch_size = gr.Slider(label="Patch size", minimum=256, maximum=1000, interactive=False)
                    self.patch_context_size = gr.Slider(label="Patch context size", minimum=256, maximum=1500, interactive=False)
                    self.patch_overlap = gr.Slider(label="Patch overlap", minimum=0, maximum=0.9, value=0.5, interactive=False)
                self.patches.change(self.update_patches_params, inputs=self.patches, outputs=[self.patch_size, self.patch_context_size, self.patch_overlap])
    
    def update_patches_params(self, enable : bool) :
        return gr.update(interactive=enable), gr.update(interactive=enable), gr.update(interactive=enable)
    
    def get_params_dict(self) :
        return {
            "alpha" : self.alpha,
            "preserve_colors" : self.pres_col,
            "resize_size" : self.res_size,
            "keep_aspect_ratio" : self.keep_ratio,
            "work_with_patches" : self.patches,
            "patch_size" : self.patch_size,
            "patch_context_size" : self.patch_context_size,
            "patch_overlap" : self.patch_overlap
        }
    

class GenConfigBlock :
    def __init__(self, scale, max_styles = 5) :
        with gr.Blocks() as self.block :
            self.max_styles = max_styles
            self.index_state = gr.State(1)
            self.style_blocks = []
            self.add_btn = gr.Button(value="➕ Add style", render=False)    
            with gr.Column(scale=scale) :
                self.content_img = gr.Image(type='filepath')
                with gr.Group() :
                    with gr.Row(equal_height=True) as row :
                        for i in range(self.max_styles):
                            self.style_blocks.append(StyleBlock(visible = (i == 0)))
                        self.link_rmv_btns()
                        self.add_btn.render() 
                        self.add_btn.click(self.show_next_block, inputs=self.index_state, outputs=[b.block for b in self.style_blocks] + [b.rmv_btn for b in self.style_blocks] + [self.index_state, self.add_btn])
                        
                self.params = ParamsBlock()

    def link_rmv_btns(self) :
        for i, block in enumerate(self.style_blocks) :
            block.rmv_btn.click(self.hide_block, inputs=[gr.State(i), self.index_state] + [b.img for b in self.style_blocks] + [b.weight for b in self.style_blocks] + [b.scale for b in self.style_blocks], outputs = [b.block for b in self.style_blocks] + [self.add_btn] + [b.rmv_btn for b in self.style_blocks] + [item for b in self.style_blocks for item in b.list()] + [self.index_state])

    def hide_block(self, index, index_state, *blocks_values) :
        new_index = index_state - 1
        img_values = blocks_values[:self.max_styles]
        weight_values = blocks_values[self.max_styles:2*self.max_styles]
        scale_values = blocks_values[2*self.max_styles:3*self.max_styles]
        visible_updates = [gr.update(visible=True) if i < new_index else gr.update(visible=False) for i in range(self.max_styles)]
        add_btn_update = gr.update(visible=(new_index < self.max_styles))
        rmv_btns_updates = [gr.update(visible=False) for _ in range(self.max_styles)] if new_index == 1 else [gr.update(visible=True) for _ in range(self.max_styles)]
        copy_updates = [[gr.update() for _ in range(3)] for _ in range(self.max_styles)]
        for i in range(index, self.max_styles - 1) :
            copy_updates[i] = self.style_blocks[i].copy(img_values[i+1], weight_values[i+1], scale_values[i+1]) #, true_res_imgs[i+1])
        copy_updates[-1] = self.style_blocks[-1].reset()
        return (*visible_updates, add_btn_update, *rmv_btns_updates, *[item for cp_update in copy_updates for item in cp_update], new_index)

    def show_next_block(self, index) :
        new_index = index + 1
        updates = [gr.update(visible = (i <= index)) for i in range(self.max_styles)]
        button_update = gr.update(visible = (new_index < self.max_styles))
        if new_index == 1 :
            rmv_btns_updates = [gr.update(visible=False) for _ in range(self.max_styles)]
        else : 
            rmv_btns_updates = [gr.update(visible=True) for _ in range(self.max_styles)]
        return (*updates, *rmv_btns_updates, new_index, button_update)
    
    def get_params_dict(self) :
        global_params = self.params.get_params_dict()
        style_images = {f"style_{i}" : self.style_blocks[i].img for i in range(len(self.style_blocks))}
        style_weights = {f"style_weight_{i}" : self.style_blocks[i].weight for i in range(len(self.style_blocks))}
        style_scales = {f"style_scale_{i}" : self.style_blocks[i].scale for i in range(len(self.style_blocks))}

        return {
            "num_visible" : self.index_state,
            "content_img" : self.content_img,
            **style_images,
            **style_weights,
            **style_scales,
            **global_params
        }

class MainInterface :
    def __init__(self, max_styles = 5):
        self.client_id = uuid.uuid4().hex
        self.max_styles = max_styles
        with gr.Blocks() as self.interface :
            with gr.Row(equal_height=False) :
                self.config = GenConfigBlock(scale=6, max_styles=self.max_styles)
                self.components_dict = self.config.get_params_dict()
                with gr.Column(scale=4) : 
                    self.generated = gr.Image(type='filepath', interactive=False)
                    self.gen_btn = gr.Button(value="Generate image")
                    self.gen_status = gr.Text(interactive=False, label="Status")
                    self.gen_msg = gr.Text(interactive=False, label="Message")
            self.gen_btn.click(self.generate_img, inputs=list(self.components_dict.values())) #, outputs=[self.generated, self.gen_status, self.gen_msg])
    
    def launch(self) :
        self.interface.launch()

In [3]:
API_URL = "ws://localhost:8000/generate"

class StyleBlock :
    def __init__(self, visible = False):
        with gr.Group(visible=visible) as self.block:
            self.img = gr.Image(interactive=True, height=300, type='filepath')
            self.weight = gr.Slider(label="Weight", minimum=0, maximum=1, value=1, interactive=True)
            self.scale = gr.Slider(label="Spatial scaling", minimum=0.1, maximum=3, value=1, interactive=True)
            self.rmv_btn = gr.Button(value="Remove style", visible=False)
            # self.true_res_img = gr.Image(render=False)
        # self.img.input(self.resize, inputs=self.img, outputs=self.img) # Image is always resized to a fixed height to make sure all style blocks are correctly aligned

    # def resize(self, img) :
    #     self.true_res_img.value = img # Keep the original image as it will be used for computation
    #     if img is None :
    #         return gr.update(value=None)
    #     # img = Image.fromarray(img)
    #     img = Image.fr
    #     new_width = int((IMG_HEIGHT / img.height) * img.width)
    #     new_img = img.resize((new_width, IMG_HEIGHT))
    #     return gr.update(value=new_img)

    def list(self) :
        return [self.img, self.weight, self.scale] #, self.true_res_img]
    
    def reset(self) :
        return (gr.update(value=None), gr.update(value=1), gr.update(value=1)) #, gr.update(value=None))
    
    def copy(self, new_img, new_weight, new_scale) : #, new_true_res_img) :
        return (gr.update(value = new_img), gr.update(value = new_weight), gr.update(value = new_scale)) #, gr.update(value = new_true_res_img))


class ParamsBlock :
    def __init__(self):
        with gr.Blocks() as self.block :
            self.alpha = gr.Slider(0, 1, value = 1, label="Importance of stylization", interactive=True)
            with gr.Row(equal_height=True) :
                with gr.Column(scale=2) :
                    self.pres_col = gr.Checkbox(label="Preserve colors", value=False)
                    # sep = gr.Markdown("---")
                    with gr.Group() :
                        self.res_size = gr.Slider(label="Resize size", minimum=256, maximum=2000, value=1000, interactive=True)
                        self.keep_ratio = gr.Checkbox(label="Keep aspect ratio", value=False)
                with gr.Column(scale=1) :
                    self.patches = gr.Checkbox(label="Work with patches", value=False)
                    self.patch_size = gr.Slider(label="Patch size", minimum=256, maximum=1000, interactive=False)
                    self.patch_context_size = gr.Slider(label="Patch context size", minimum=256, maximum=1500, interactive=False)
                    self.patch_overlap = gr.Slider(label="Patch overlap", minimum=0, maximum=0.9, value=0.5, interactive=False)
                self.patches.change(self.update_patches_params, inputs=self.patches, outputs=[self.patch_size, self.patch_context_size, self.patch_overlap])
    
    def update_patches_params(self, enable : bool) :
        return gr.update(interactive=enable), gr.update(interactive=enable), gr.update(interactive=enable)
    
    def get_params_dict(self) :
        return {
            "alpha" : self.alpha,
            "preserve_colors" : self.pres_col,
            "resize_size" : self.res_size,
            "keep_aspect_ratio" : self.keep_ratio,
            "work_with_patches" : self.patches,
            "patch_size" : self.patch_size,
            "patch_context_size" : self.patch_context_size,
            "patch_overlap" : self.patch_overlap
        }
    

class GenConfigBlock :
    def __init__(self, scale, max_styles = 5) :
        with gr.Blocks() as self.block :
            self.max_styles = max_styles
            self.index_state = gr.State(1)
            self.style_blocks = []
            self.add_btn = gr.Button(value="➕ Add style", render=False)    
            with gr.Column(scale=scale) :
                self.content_img = gr.Image(type='filepath')
                with gr.Group() :
                    with gr.Row(equal_height=True) as row :
                        for i in range(self.max_styles):
                            self.style_blocks.append(StyleBlock(visible = (i == 0)))
                        self.link_rmv_btns()
                        self.add_btn.render() 
                        self.add_btn.click(self.show_next_block, inputs=self.index_state, outputs=[b.block for b in self.style_blocks] + [b.rmv_btn for b in self.style_blocks] + [self.index_state, self.add_btn])
                        
                self.params = ParamsBlock()

    def link_rmv_btns(self) :
        for i, block in enumerate(self.style_blocks) :
            block.rmv_btn.click(self.hide_block, inputs=[gr.State(i), self.index_state] + [b.img for b in self.style_blocks] + [b.weight for b in self.style_blocks] + [b.scale for b in self.style_blocks], outputs = [b.block for b in self.style_blocks] + [self.add_btn] + [b.rmv_btn for b in self.style_blocks] + [item for b in self.style_blocks for item in b.list()] + [self.index_state])

    def hide_block(self, index, index_state, *blocks_values) :
        new_index = index_state - 1
        img_values = blocks_values[:self.max_styles]
        weight_values = blocks_values[self.max_styles:2*self.max_styles]
        scale_values = blocks_values[2*self.max_styles:3*self.max_styles]
        visible_updates = [gr.update(visible=True) if i < new_index else gr.update(visible=False) for i in range(self.max_styles)]
        add_btn_update = gr.update(visible=(new_index < self.max_styles))
        rmv_btns_updates = [gr.update(visible=False) for _ in range(self.max_styles)] if new_index == 1 else [gr.update(visible=True) for _ in range(self.max_styles)]
        copy_updates = [[gr.update() for _ in range(3)] for _ in range(self.max_styles)]
        for i in range(index, self.max_styles - 1) :
            copy_updates[i] = self.style_blocks[i].copy(img_values[i+1], weight_values[i+1], scale_values[i+1]) #, true_res_imgs[i+1])
        copy_updates[-1] = self.style_blocks[-1].reset()
        return (*visible_updates, add_btn_update, *rmv_btns_updates, *[item for cp_update in copy_updates for item in cp_update], new_index)

    def show_next_block(self, index) :
        new_index = index + 1
        updates = [gr.update(visible = (i <= index)) for i in range(self.max_styles)]
        button_update = gr.update(visible = (new_index < self.max_styles))
        if new_index == 1 :
            rmv_btns_updates = [gr.update(visible=False) for _ in range(self.max_styles)]
        else : 
            rmv_btns_updates = [gr.update(visible=True) for _ in range(self.max_styles)]
        return (*updates, *rmv_btns_updates, new_index, button_update)
    
    def get_params_dict(self) :
        global_params = self.params.get_params_dict()
        style_images = {f"style_{i}" : self.style_blocks[i].img for i in range(len(self.style_blocks))}
        style_weights = {f"style_weight_{i}" : self.style_blocks[i].weight for i in range(len(self.style_blocks))}
        style_scales = {f"style_scale_{i}" : self.style_blocks[i].scale for i in range(len(self.style_blocks))}

        return {
            "num_visible" : self.index_state,
            "content_img" : self.content_img,
            **style_images,
            **style_weights,
            **style_scales,
            **global_params
        }

class MainInterface :
    def __init__(self, max_styles = 5):
        self.client_id = uuid.uuid4().hex
        self.max_styles = max_styles
        with gr.Blocks() as self.interface :
            with gr.Row(equal_height=False) :
                self.config = GenConfigBlock(scale=6, max_styles=self.max_styles)
                self.components_dict = self.config.get_params_dict()
                with gr.Column(scale=4) : 
                    self.generated = gr.Image(type='filepath', interactive=False)
                    self.gen_btn = gr.Button(value="Generate image")
                    self.gen_status = gr.Text(interactive=False, label="Status")
                    self.gen_msg = gr.Text(interactive=False, label="Message")
            self.gen_btn.click(self.generate_img, inputs=list(self.components_dict.values())) #, outputs=[self.generated, self.gen_status, self.gen_msg])
    
    def launch(self) :
        self.interface.launch()
    
    async def generate_img(self, *values) :
        keys = list(self.components_dict.keys())
        params = {key : value for key, value in zip(keys, values)}
        num_visible = params["num_visible"]
        content_img = Path(params["content_img"])
        actual_style_imgs_ixs = [i for i in range(num_visible) if params[f"style_{i}"] is not None]
        style_imgs = [Path(params[f"style_{i}"]) for i in actual_style_imgs_ixs]
        style_weights = [params[f"style_weight_{i}"] for i in actual_style_imgs_ixs]
        style_scales = [params[f"style_scale_{i}"] for i in actual_style_imgs_ixs]
        gen_param_keys = ["alpha", "preserve_colors", "resize_size", "keep_aspect_ratio", "work_with_patches", "patch_size", "patch_context_size", "patch_overlap"]
        gen_params = {
            "style_weights" : style_weights,
            "style_scales" : style_scales,
            **{key : params[key] for key in gen_param_keys}
        }
        request = {
            "client_id" : self.client_id,
            "content" : File.from_path(content_img).model_dump(),
            "style" : [File.from_path(style_img).model_dump() for style_img in style_imgs],
            "params" : gen_params
        }
        async with connect(API_URL, max_size=5*1024*1024) as websocket :
            task = await asyncio.create_task(self.send_request(websocket, request))
            # response = await request_generation(websocket, self.client_id, content_img, style_imgs, params)

    async def send_request(self, connection, request) :
        try :
            await connection.send(json.dumps(request))
            try:
                response = await connection.recv()
                msg = json.loads(response)
                # status_update = gr.update(value=msg.get("status"))
                # msg_update = gr.update(value=msg.get("message"))
                if msg.get("status") in ("success"):
                    gen_file = File.from_dict(msg["generated_image"])
                    gen_img_path = gen_file.save_to(Path('D:/StyleTransferAI/StyleTransferAI_AdaIN/StyleTransferAI-based-on-AdaIN/style_transfer_api/tmp/resp'))
                # gen_img_update = gr.update(value=gen_img_path)
                # return (gen_img_update, status_update, msg_update)
                # self.generated.update(value=gen_img_path)
                gr.update(self.generated.elem_id, value=gen_img_path)
                # self.gen_status.update(value=msg.get("status"))
                gr.update(self.gen_status.elem_id, value=msg.get("status"))
                # self.gen_msg.update(value=msg.get("message"))
                gr.update(self.gen_msg.elem_id, value=msg.get("message"))
            except websockets.ConnectionClosed:
                # return (gr.update(value=None), gr.update(value="Connection closed"), gr.update(value=None))
                # self.generated.update(value=None)
                gr.update(self.generated.elem_id, value=None)
                # self.gen_status.update(value="Connection closed")
                gr.update(self.gen_status.elem_id, value="Connection closed")
                # self.gen_msg.update(value=None)
                gr.update(self.gen_msg.elem_id, value=None)
        except Exception as e :
            print(e)

async def request_generation(connection, client_id, content_img, style_imgs, params) :
    request = Request(client_id=client_id, content_img=content_img, style_imgs=style_imgs, params=params)
    await connection.send(request.model_dump())
    response = await Response(**connection.recv())
    return Response

interface = MainInterface()
interface.launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [2]:
{'a' : 2} | {'b' : 4}

{'a': 2, 'b': 4}